### Combine the data and prep it for training

In [1]:
import os
import random
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import datasets

DATA = "../data/"
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

CHUNK = 40        # lines per chunk
MIN_LINES = 500   # poets with fewer different lines than this become "Unknown"
MAX_TOKENS = 500  # a bit under 512 so the header swap later can't push it over
DROPOUT = 0.15    # share of named train chunks that get "Unknown" instead
SEED = 42

os.makedirs(DATA + "train_test", exist_ok=True)

c:\Users\shubh\Desktop\Academics\Computational Creativity\Assignment 2\Bad_Poets_Society\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gutenberg = pd.read_parquet(DATA + "gutenberg_clean.parquet")
pdp = pd.read_parquet(DATA + "pdp_clean.parquet")
print(len(gutenberg), "Gutenberg books")
print(len(pdp), "PDP poems")

869 Gutenberg books
34245 PDP poems


In [3]:
def clean_line(s):
    return re.sub(r"[^a-z]", "", s.lower())

# keep the pdp copy of a poem (it knows where the poem ends) and take those lines out of gutenberg instead
pdp_lines = set()
for text in pdp['text']:
    for line in text.split('\n'):
        cl = clean_line(line)
        if len(cl) >= 20:
            pdp_lines.add(cl)

def in_pdp(line):
    cl = clean_line(line)
    return len(cl) >= 20 and cl in pdp_lines

def remove_pdp_lines(text):
    lines = text.split('\n')
    keep = []
    for i in range(0, len(lines), CHUNK):
        window = lines[i:i+CHUNK]
        # if any line of the window is in pdp, drop the whole window. taking out just the matching lines
        # leaves holes (other editions spell some lines differently so only some of them match)
        if any(in_pdp(l) for l in window):
            continue
        keep += window
    return '\n'.join(keep)

def count_lines(text):
    return len(text.split('\n')) if text else 0

before = gutenberg['text'].apply(count_lines).sum()
gutenberg['text'] = gutenberg['text'].apply(remove_pdp_lines)

# whitman's gutenberg book has his long lines wrapped so they never match, just use his pdp poems
gutenberg = gutenberg[gutenberg['author'] != 'Walt Whitman']
gutenberg['n_lines'] = gutenberg['text'].apply(count_lines)
gutenberg = gutenberg[gutenberg['n_lines'] >= 4]

print("Removed", before - gutenberg['n_lines'].sum(), "Gutenberg lines that are also in PDP (+ Whitman)")
print(len(gutenberg), "Gutenberg books left,", gutenberg['n_lines'].sum(), "lines")

Removed 939605 Gutenberg lines that are also in PDP (+ Whitman)
620 Gutenberg books left, 990542 lines


In [4]:
# every poet with 500+ different lines keeps their name, everyone else is "Unknown"
poet_lines = {}
for df in [pdp, gutenberg]:
    for author, text in zip(df['author'], df['text']):
        if author == 'Unknown':
            continue
        poet_lines.setdefault(author, set()).update(clean_line(l) for l in text.split('\n') if clean_line(l))

named = pd.Series({a: len(s) for a, s in poet_lines.items()}).sort_values(ascending=False)
named = named[named >= MIN_LINES]
named.rename('distinct_lines').to_csv(DATA + 'train_test/named_poets.csv', index_label='author')

pdp['author'] = pdp['author'].where(pdp['author'].isin(named.index), 'Unknown')
gutenberg['author'] = gutenberg['author'].where(gutenberg['author'].isin(named.index), 'Unknown')

print(len(named), "poets keep their name")
named.head(10)

486 poets keep their name


Robert Browning               57456
William Wordsworth            54597
William Cowper                53696
Madison Julius Cawein         49039
Algernon Charles Swinburne    44693
Henry Wadsworth Longfellow    43514
Thomas Moore                  41550
George Gordon Byron           38165
Alfred Lord Tennyson          38133
John Greenleaf Whittier       36764
dtype: int64

In [5]:
# chunk each poem / book on its own so chunks never mix two poems
chunks = []
for source, df in [('pdp', pdp), ('gutenberg', gutenberg)]:
    for doc_id, author, text in zip(df.index, df['author'], df['text']):
        lines = text.split('\n')
        for i in range(0, len(lines), CHUNK):
            chunks.append({
                'source': source,
                'doc_id': f"{source}_{doc_id}",  # poem 19 and book 19 are not the same thing
                'author': author,
                'lines': lines[i:i+CHUNK],
                'is_end': source == 'pdp' and i + CHUNK >= len(lines),  # last chunk of a poem
            })

chunks_df = pd.DataFrame(chunks)
print(f"Created {len(chunks_df)} chunks")

Created 93716 chunks


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_text(author, lines, is_end):
    text = f"Author: {author}\n\n" + "\n".join(lines)
    if is_end:
        text += tokenizer.eos_token  # only the end of a poem gets the end-of-text token
    return text

def n_tokens(row):
    return len(tokenizer(make_text(row['author'], row['lines'], row['is_end']))['input_ids'])

# some chunks have very long lines (whitman etc), cut those in half until they fit
dropped = 0
def fit(row):
    global dropped
    if n_tokens(row) <= MAX_TOKENS:
        return [row]
    if len(row['lines']) == 1:  # one line on its own is too long, can't cut it so drop it
        dropped += 1
        return []
    half = len(row['lines']) // 2
    return fit(dict(row, lines=row['lines'][:half], is_end=False)) + fit(dict(row, lines=row['lines'][half:]))

texts = [make_text(a, l, e) for a, l, e in zip(chunks_df['author'], chunks_df['lines'], chunks_df['is_end'])]
chunks_df['n_tokens'] = [len(x) for x in tokenizer(texts)['input_ids']]
long_rows = chunks_df[chunks_df['n_tokens'] > MAX_TOKENS]
fixed = [r for row in long_rows.to_dict('records') for r in fit(row)]
chunks_df = pd.concat([chunks_df[chunks_df['n_tokens'] <= MAX_TOKENS], pd.DataFrame(fixed)], ignore_index=True)

print(len(long_rows), "chunks were too long and got cut up,", dropped, "single lines dropped")

5450 chunks were too long and got cut up, 32 single lines dropped


In [7]:
# split by poem / book (not by chunk) so no poem ends up in both train and test
docs = chunks_df['doc_id'].unique().tolist()
train_ids, temp_ids = train_test_split(docs, test_size=0.1, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)

train_df = chunks_df[chunks_df['doc_id'].isin(train_ids)].copy()
val_df = chunks_df[chunks_df['doc_id'].isin(val_ids)].copy()
test_df = chunks_df[chunks_df['doc_id'].isin(test_ids)].copy()

print(f"Train   : {len(train_df)} chunks from {len(train_ids)} poems/books")
print(f"Val     : {len(val_df)} chunks from {len(val_ids)} poems/books")
print(f"Test    : {len(test_df)} chunks from {len(test_ids)} poems/books")

Train   : 89344 chunks from 31378 poems/books
Val     : 5788 chunks from 1743 poems/books
Test    : 5067 chunks from 1744 poems/books


In [8]:
# 15% of the named train chunks get "Unknown" so the model can also write without a poet (train only)
rng = random.Random(SEED)
def header_author(author):
    if author != 'Unknown' and rng.random() < DROPOUT:
        return 'Unknown'
    return author

train_df['header'] = [header_author(a) for a in train_df['author']]
val_df['header'] = val_df['author']
test_df['header'] = test_df['author']

for d in [train_df, val_df, test_df]:
    d['text'] = [make_text(h, l, e) for h, l, e in zip(d['header'], d['lines'], d['is_end'])]

named_train = train_df[train_df['author'] != 'Unknown']
print(f"{(named_train['header'] == 'Unknown').mean():.1%} of named train chunks got Unknown")

15.2% of named train chunks got Unknown


In [9]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

cols = ['source', 'doc_id', 'author', 'header', 'is_end', 'text']
ds = datasets.DatasetDict({
    'train': datasets.Dataset.from_pandas(train_df[cols], preserve_index=False),
    'val': datasets.Dataset.from_pandas(val_df[cols], preserve_index=False),
    'test': datasets.Dataset.from_pandas(test_df[cols], preserve_index=False)
})

tokenized_ds = ds.map(tokenize_function, batched=True)

tokenized_ds['train'].to_parquet(DATA + 'train_test/' + 'train.parquet')
tokenized_ds['val'].to_parquet(DATA   + 'train_test/' + 'val.parquet')
tokenized_ds['test'].to_parquet(DATA  + 'train_test/' + 'test.parquet')
print("Saved train, val, and test datasets to data/train_test/")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  3.69ba/s]

Saved train, val, and test datasets to data/train_test/
